# AeroGuard Notebook 02: Preprocessing & Leakage-Free Splitting
Identifying constant sensors, engine-wise train/validation partitioning, and scaling.


In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from src.data_loader import load_raw_dataset
from src.rul_calculation import calculate_train_rul
from src.preprocessing import CMAPSSPreprocessor
from src.model_training import split_engine_wise

df_train, _, _ = load_raw_dataset("FD001")
df_train_rul = calculate_train_rul(df_train, max_rul_clip=125)

# Engine-wise split to avoid cross-record data leakage
df_tr_split, df_val_split, tr_ids, val_ids = split_engine_wise(df_train_rul, val_ratio=0.2)
print(f"Train engines: {len(tr_ids)} ({len(df_tr_split)} cycles)")
print(f"Validation engines: {len(val_ids)} ({len(df_val_split)} cycles)")


In [ ]:
# Fit preprocessor strictly on training fold
preprocessor = CMAPSSPreprocessor(scaler_type="standard")
preprocessor.fit(df_tr_split)

print("Constant/Invariant Channels Dropped:", preprocessor.constant_columns_)
print("Informative Telemetry Channels Retained:", len(preprocessor.active_sensor_cols_))

df_tr_scaled = preprocessor.transform(df_tr_split)
df_val_scaled = preprocessor.transform(df_val_split)
df_tr_scaled.head()
